# Experimentos: ATP y fuerza externa

Estudia cómo la concentración de ATP y una carga opuesta afectan las tasas y los observables emergentes de muchas trayectorias simuladas.

> **Nota científica:** ProteinMotorSim es un prototipo de modelado cinético fenomenológico. Los parámetros iniciales son ilustrativos y los mapas de energía no están calibrados experimentalmente para kinesina.

In [ ]:
from pathlib import Path
import sys

# Busca la raíz del proyecto para poder importar src/protein_motor_sim.
PROJECT_ROOT = None
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "src" / "protein_motor_sim").is_dir():
        PROJECT_ROOT = candidate
        sys.path.insert(0, str(candidate / "src"))
        break

if PROJECT_ROOT is None:
    raise RuntimeError(
        "No encuentro la carpeta src/protein_motor_sim. "
        "Abre JupyterLab desde la raíz de ProteinMotorSim o instala el paquete "
        "con python -m pip install -e ."
    )

print(f"Proyecto detectado en: {PROJECT_ROOT}")

## 1. Preguntas del experimento

1. ¿Cómo cambia la tasa de unión de ATP con la concentración?
2. ¿Cómo cambia el comportamiento simulado al aumentar una carga opuesta?

Separaremos la ley cinética elemental de los resultados emergentes del conjunto de simulaciones.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from dataclasses import replace

from protein_motor_sim import Environment, GillespieSimulator, Kinesin1
from protein_motor_sim.kinetics.rate_laws import ATPBindingRate
from protein_motor_sim.kinetics.force_dependence import BellRate
from protein_motor_sim.kinetics.rate_laws import ConstantRate
from protein_motor_sim.simulation.ensemble import run_ensemble
from protein_motor_sim.analysis.statistics import summarize_trajectories
from protein_motor_sim.analysis.parameter_sweep import sweep_atp_concentration

## 2. Efecto directo de ATP sobre la tasa de asociación

Para una asociación bimolecular aproximada, la tasa pseudo-primer orden es `k_ATP = k_on [ATP]`. Esta relación describe solo la etapa de unión, no la velocidad global del motor.

In [ ]:
k_on = 1.0e6  # M^-1 s^-1, valor ilustrativo
rate_law = ATPBindingRate(association_rate_M_s=k_on)
atp_M = np.logspace(-6, -2, 100)
rates_s = np.array([rate_law(Environment(atp_concentration=c)) for c in atp_M])

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(atp_M * 1e3, rates_s)
ax.set_xlabel("[ATP] (mM)")
ax.set_ylabel("Tasa de asociación (s$^{-1}$)")
ax.set_title("Ley de asociación de ATP")
ax.grid(True, which="both", alpha=0.25)
plt.show()

## 3. Barrido de concentración de ATP

Ejecutamos múltiples realizaciones para cada concentración. La media y desviación estándar describen la dispersión entre trayectorias, no la incertidumbre experimental.

In [ ]:
motor = Kinesin1()
simulator = GillespieSimulator()
base_environment = Environment(temperature_K=310.0, external_force_N=0.0)
concentrations_M = np.array([0.05, 0.1, 0.25, 0.5, 1.0, 2.0]) * 1e-3

atp_results = sweep_atp_concentration(
    simulator=simulator,
    motor=motor,
    environment=base_environment,
    concentrations_M=concentrations_M,
    t_max=0.7,
    n_runs=30,
    seed=120,
)
atp_df = pd.DataFrame(atp_results)
atp_df

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(
    atp_df["atp_concentration_M"] * 1e3,
    atp_df["mean_velocity_nm_s"],
    yerr=atp_df["std_velocity_nm_s"],
    marker="o", capsize=3,
)
ax.set_xlabel("[ATP] (mM)")
ax.set_ylabel("Velocidad media neta (nm/s)")
ax.set_title("Respuesta simulada frente a ATP")
ax.grid(alpha=0.25)
plt.show()

## 4. Efecto de la fuerza opuesta

La clase `BellRate` modifica las tasas según la fuerza. En el modelo actual, la carga reduce la tasa de avance, aumenta la tasa de retroceso y puede aumentar el desprendimiento según la convención elegida en el modelo. Esto es una parametrización fenomenológica; no es todavía una curva fuerza–velocidad calibrada.

In [ ]:
forces_pN = np.array([0.0, 1.0, 2.0, 3.0, 4.0, 5.0])
force_rows = []
for force_pN in forces_pN:
    condition = Environment(
        atp_concentration=1e-3,
        external_force_N=force_pN * 1e-12,
        temperature_K=310.0,
    )
    ensemble = run_ensemble(
        simulator, motor, condition, t_max=0.7, n_runs=30, seed=500 + int(force_pN * 10)
    )
    stats = summarize_trajectories(ensemble.trajectories)
    force_rows.append({
        "force_pN": force_pN,
        "mean_velocity_nm_s": stats.mean_velocity_nm_s,
        "std_velocity_nm_s": stats.std_velocity_nm_s,
        "mean_run_length_nm": stats.mean_run_length_nm,
    })
force_df = pd.DataFrame(force_rows)
force_df

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(
    force_df["force_pN"], force_df["mean_velocity_nm_s"],
    yerr=force_df["std_velocity_nm_s"], marker="o", capsize=3,
)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("Fuerza opuesta (pN)")
ax.set_ylabel("Velocidad media neta (nm/s)")
ax.set_title("Respuesta simulada frente a la carga")
ax.grid(alpha=0.25)
plt.show()

## 5. Interpretación y límites

- Aumentar ATP acelera la asociación de ATP, pero otras etapas pueden limitar el ciclo.
- La fuerza cambia tasas particulares según el modelo de Bell; no se ha impuesto una descripción termodinámica completa.
- La velocidad reportada es la velocidad neta de la trayectoria hasta el final de observación o desprendimiento.
- No interpretes estos resultados como parámetros experimentales ni como predicción cuantitativa de la kinesina real sin calibración.